In [1]:
# realizando todos os imports necessários para execução do pipeline
import os
import json
import pickle
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
import optuna
from optuna.samplers import TPESampler

In [2]:
load_dotenv()

True

In [3]:
# lendo as variáveis de ambiente para definir os caminhos dos diretórios principais
data_path = os.getenv("DATA_PATH")
models_path = os.getenv("MODELS_PATH")
results_path = os.getenv("RESULTS_PATH")

In [4]:
# importando pacote para realizar chamadas no sistema operacional
import subprocess


# criando função que checa se o comando nvidia-smi funciona, indicando presença de gpu
def cuda_available():
    try:
        subprocess.check_output(["nvidia-smi"])
        return True
    except Exception:
        return False


# setando a variável que será repassada aos modelos para garantir uso do hardware correto
DEVICE = "cuda" if cuda_available() else "cpu"
print(f"dispositivo: {DEVICE}")

dispositivo: cpu


In [5]:
# carregando o dataset com os dados que vão alimentar o modelo 2
df = pd.read_parquet(os.path.join(data_path, "model2_dataset.parquet"))
df.head()

,player_id,transfer_date,transfer_fee,predicted_value_m1,buyer_ratio,seller_ratio,buyer_league_tier,seller_league_tier,same_confederation,transfer_window,...,national_team_ranking_inv,goals_per_90,assists_per_90,buyer_net_transfer_record,buyer_national_team_players,buyer_stadium_seats,seller_net_transfer_record,seller_national_team_players,seller_stadium_seats,log_transfer_fee
0,81796,2010-01-01,2400000.0,2.643977e+06,1.000000,1.000000,4.0,5.0,0,1,...,0.0,0.0,0.0,14570000.0,15.0,60411.0,1260000.0,3.0,66704.0,14.690980
1,66515,2010-01-01,6000000.0,2.100220e+06,1.000000,1.000000,4.0,5.0,0,1,...,0.0,0.0,0.0,10000000.0,11.0,34725.0,23850000.0,5.0,24584.0,15.607270
2,42457,2010-01-01,600000.0,2.190705e+06,1.000000,1.000000,5.0,4.0,0,1,...,0.0,0.0,0.0,605000.0,1.0,28678.0,280000.0,11.0,55634.0,13.304687
3,33829,2010-01-09,1400000.0,2.461618e+06,0.907724,0.907724,1.0,1.0,1,1,...,0.0,0.0,0.0,86560000.0,6.0,32384.0,-47850000.0,20.0,25486.0,14.151984
4,38746,2010-01-11,800000.0,8.720824e+05,0.738228,0.738228,4.0,1.0,1,1,...,0.0,0.0,0.0,4790000.0,2.0,6108.0,-182900000.0,19.0,62850.0,13.592368


In [6]:
# listando as colunas que não devem ser usadas como variável explicativa no treinamento
NON_FEATURES = [
    "player_id",
    "buyer_club_id",
    "seller_club_id",
    "transfer_date",
    "transfer_fee",
    "log_transfer_fee",
]

# filtrando o dataset para manter apenas as colunas que são de fato features
FEATURES = [c for c in df.columns if c not in NON_FEATURES]
FEATURES

['predicted_value_m1',
 'buyer_ratio',
 'seller_ratio',
 'buyer_league_tier',
 'seller_league_tier',
 'same_confederation',
 'transfer_window',
 'transfer_year',
 'player_age_at_transfer',
 'position_rank',
 'national_team_ranking_inv',
 'goals_per_90',
 'assists_per_90',
 'buyer_net_transfer_record',
 'buyer_national_team_players',
 'buyer_stadium_seats',
 'seller_net_transfer_record',
 'seller_national_team_players',
 'seller_stadium_seats']

In [7]:
# ordenando cronologicamente os dados de transferência para evitar vazamento futuro
df = df.sort_values("transfer_date").reset_index(drop=True)

# separando os dados em bases de treino e teste através de split temporal
df_train = df[df["transfer_date"] < "2022-01-01"].copy()
df_test = df[df["transfer_date"] >= "2022-01-01"].copy()

# criando as variáveis explicativas (X) e o target em log (y) para ambas as bases
X_train, y_train = df_train[FEATURES].astype(float), df_train["log_transfer_fee"]
X_test, y_test = df_test[FEATURES].astype(float), df_test["log_transfer_fee"]


In [8]:
# inicializando o gerador de k-folds com k=5 para orientar a validação do optuna
kf = TimeSeriesSplit(n_splits=5)


# declarando função objetivo que treina o modelo iterativamente para o optuna
def objective(trial):
    # buscando os melhores parametros
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 300, 1200),
        max_depth=trial.suggest_int("max_depth", 4, 7),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 5, 20),
        gamma=trial.suggest_float("gamma", 0.0, 2.0),
        objective="reg:quantileerror",
        quantile_alpha=0.50,
        random_state=42,
        tree_method="hist",
        device=DEVICE,
        n_jobs=-1,
    )

    # declarando lista para acumular o erro quadrático médio em cada iteração do fold
    rmse_folds = []

    # iterando sobre cada partição separando índices de treino e validação gerados pelo kfold
    for tr_idx, cv_idx in kf.split(X_train):
        # dividindo os dataframes de treino e validação para este fold específico
        X_tr, X_cv = X_train.iloc[tr_idx], X_train.iloc[cv_idx]
        y_tr, y_cv = y_train.iloc[tr_idx], y_train.iloc[cv_idx]

        # instanciando o modelo usando os parâmetros sugeridos com parada antecipada
        m = XGBRegressor(**params, early_stopping_rounds=30)

        # executando treinamento do modelo na base de treino provisória
        m.fit(X_tr, y_tr, eval_set=[(X_cv, y_cv)], verbose=False)

        # calculando a raiz quadrada do erro quadrático médio e adicionando à lista
        rmse_folds.append(np.sqrt(np.mean((y_cv.values - m.predict(X_cv)) ** 2)))

    # retornando a média do erro avaliado nos 5 folds para ser interpretada pelo optuna
    return float(np.mean(rmse_folds))

In [9]:
# criando e executando o estudo do optuna focando em minimizar o rmse  (mediana)
study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100)

# recuperando e armazenando o melhor dicionário de hiperparâmetros encontrado pelo optuna
best_params = study.best_params

# adicionando parâmetros fixos complementares na configuração ótima antes do treinamento final
best_params["objective"] = "reg:quantileerror"
best_params["tree_method"] = "hist"
best_params["device"] = DEVICE
best_params["random_state"] = 42
best_params["n_jobs"] = -1
print("Melhores parâmetros encontrados:", best_params)

[I 2026-07-18 18:57:00,464] A new study created in memory with name: no-name-c5a01c1c-5414-41ce-b1fb-908d264fa1b3


[I 2026-07-18 18:57:04,432] Trial 0 finished with value: 0.6282310476950942 and parameters: {'n_estimators': 637, 'max_depth': 7, 'learning_rate': 0.07259248719561363, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 0.029375384576328288, 'reg_lambda': 0.13066739238053282, 'min_child_weight': 18, 'gamma': 1.2022300234864176}. Best is trial 0 with value: 0.6282310476950942.
[I 2026-07-18 18:57:05,447] Trial 1 finished with value: 0.63755265573188 and parameters: {'n_estimators': 937, 'max_depth': 4, 'learning_rate': 0.13826189316223852, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.035113563139704075, 'reg_lambda': 0.2327067708383781, 'min_child_weight': 9, 'gamma': 1.0495128632644757}. Best is trial 0 with value: 0.6282310476950942.
[I 2026-07-18 18:57:08,063] Trial 2 finished with value: 0.6223649322258238 and parameters: {'n_estimators': 689, 'max_depth': 5, 'learning_rate': 0.05243180891902853, 'subsample

Melhores parâmetros encontrados: {'n_estimators': 1135, 'max_depth': 6, 'learning_rate': 0.015056802694342578, 'subsample': 0.7009334442953915, 'colsample_bytree': 0.8247227112122452, 'reg_alpha': 0.8595903116845056, 'reg_lambda': 0.881149188117154, 'min_child_weight': 13, 'gamma': 0.2849237870421557, 'objective': 'reg:quantileerror', 'tree_method': 'hist', 'device': 'cpu', 'random_state': 42, 'n_jobs': -1}


In [10]:
# criando versão do modelo xgboost que vai estimar o quantil 15% inferior (p15)
model_p15 = XGBRegressor(**best_params, quantile_alpha=0.15)

# criando versão do modelo xgboost que vai estimar o valor mediano, quantil 50% (p50)
model_p50 = XGBRegressor(**best_params, quantile_alpha=0.50)

# criando versão do modelo xgboost que vai estimar o quantil 85% superior (p85)
model_p85 = XGBRegressor(**best_params, quantile_alpha=0.85)

In [11]:
# executando o ajuste/treinamento dos 3 modelos paralelamente na base total de treinamento
model_p15.fit(X_train, y_train)
model_p50.fit(X_train, y_train)
model_p85.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8247227112122452, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.2849237870421557,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.015056802694342578,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=13, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1135, n_jobs=-1,
             num_parallel_tree=None, objective='reg:quantileerror', ...)

In [12]:
# registrando em formato pickle a lista com os nomes exatos das features utilizadas na modelagem
with open(os.path.join(models_path, "feature_names_model2.pkl"), "wb") as f:
    pickle.dump(FEATURES, f)

In [13]:
# gravando em disco o primeiro modelo para projeção do limite inferior de preço (p15)
model_p15.save_model(os.path.join(models_path, "model2_transfer_fee_p15.json"))

# gravando em disco o segundo modelo central com base no comportamento mediano (p50)
model_p50.save_model(os.path.join(models_path, "model2_transfer_fee_p50.json"))

# gravando em disco o terceiro e último modelo que define o teto do intervalo estimado (p85)
model_p85.save_model(os.path.join(models_path, "model2_transfer_fee_p85.json"))